# External model — streaming completion

OpenAI-compatible **`POST …/v1/completions`** (or same host) with **`stream: true`**. Response is read as **SSE** and printed as tokens arrive.

**Prereq:** Python 3.9+ (stdlib: `json`, `ssl`, `urllib`).

**Credentials:** Same pattern as **`maas-validation-demo-with-key.ipynb`**: `MAAS_API_KEY` / `API_KEY` / **Demo quick swap** for the bearer token. **`VERIFY_TLS`** from the environment (`1` / `true` for strict TLS).

**Model target:** Set **`EXTERNAL_MODEL_NAME`** and **`EXTERNAL_COMPLETIONS_URL`** in the next section (full HTTPS URL to the completions endpoint you are demoing).

## Demo quick swap

Optional paste between takes. Run this cell, then **Hardcoded model target**.

| Variable | Effect |
|----------|--------|
| `DEMO_API_KEY` | Non-empty → overrides `MAAS_API_KEY` / `API_KEY` env. |
| `DEMO_EXTERNAL_MODEL_NAME` | Non-empty → overrides the hardcoded model id below. |
| `DEMO_EXTERNAL_COMPLETIONS_URL` | Non-empty → overrides the hardcoded completions URL below. |

In [ ]:
DEMO_API_KEY = ""
DEMO_EXTERNAL_MODEL_NAME = ""
DEMO_EXTERNAL_COMPLETIONS_URL = ""

## Hardcoded model target

Edit **`EXTERNAL_MODEL_NAME`** and **`EXTERNAL_COMPLETIONS_URL`** for the external route you are showing (full URL, including `/v1/completions` if that is your API shape).

In [ ]:
# --- edit for the model you are demoing ---
EXTERNAL_MODEL_NAME = "your-provider/your-model-id"
EXTERNAL_COMPLETIONS_URL = "https://your-endpoint.example.com/v1/completions"

_dm = globals().get("DEMO_EXTERNAL_MODEL_NAME", "")
if isinstance(_dm, str) and _dm.strip():
    EXTERNAL_MODEL_NAME = _dm.strip()
_du = globals().get("DEMO_EXTERNAL_COMPLETIONS_URL", "")
if isinstance(_du, str) and _du.strip():
    EXTERNAL_COMPLETIONS_URL = _du.strip()

import json
import os
import ssl
import urllib.error
import urllib.request

_ak = globals().get("DEMO_API_KEY", "")
if isinstance(_ak, str) and _ak.strip():
    API_KEY = _ak.strip()
else:
    API_KEY = (os.environ.get("MAAS_API_KEY") or os.environ.get("API_KEY") or "").strip()

VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")

if not API_KEY:
    raise SystemExit("Set DEMO_API_KEY or MAAS_API_KEY / API_KEY.")

print("Model id :", EXTERNAL_MODEL_NAME)
print("POST URL :", EXTERNAL_COMPLETIONS_URL)
print("VERIFY_TLS:", VERIFY_TLS)
print("API key set:", bool(API_KEY))

## Stream one completion

Sends **`stream: true`**, reads **`text/event-stream`** (SSE), prints token deltas to stdout, then a short summary.

In [ ]:
PROMPT = "Say hello in one short sentence."
MAX_TOKENS = 256


def _ssl_ctx():
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    return ctx


def stream_openai_compatible_completions(
    url: str,
    *,
    api_key: str,
    model: str,
    prompt: str,
    max_tokens: int,
) -> None:
    payload = {
        "model": model,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "stream": True,
    }
    body = json.dumps(payload).encode("utf-8")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }
    req = urllib.request.Request(url, data=body, headers=headers, method="POST")
    ctx = _ssl_ctx()
    total_chars = 0
    last_usage = None
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=300) as resp:
            while True:
                line = resp.readline()
                if not line:
                    break
                s = line.decode("utf-8", errors="replace").strip()
                if not s or s.startswith(":"):
                    continue
                if not s.startswith("data: "):
                    continue
                data = s[6:].strip()
                if data == "[DONE]":
                    break
                try:
                    obj = json.loads(data)
                except json.JSONDecodeError:
                    continue
                if isinstance(obj, dict) and "usage" in obj:
                    last_usage = obj.get("usage")
                for ch in obj.get("choices") or []:
                    if not isinstance(ch, dict):
                        continue
                    t = ch.get("text")
                    if t:
                        print(t, end="", flush=True)
                        total_chars += len(t)
                    delta = ch.get("delta")
                    if isinstance(delta, dict):
                        c = delta.get("content")
                        if c:
                            print(c, end="", flush=True)
                            total_chars += len(c)
    except urllib.error.HTTPError as e:
        err = e.read().decode("utf-8", errors="replace")
        print("HTTP", e.code, err[:2000])
        return
    print()
    print("---")
    print("Stream finished. Approx printed chars:", total_chars)
    if last_usage:
        print("usage:", last_usage)


stream_openai_compatible_completions(
    EXTERNAL_COMPLETIONS_URL,
    api_key=API_KEY,
    model=EXTERNAL_MODEL_NAME,
    prompt=PROMPT,
    max_tokens=MAX_TOKENS,
)